# Fine tuning, for the n=4 case

In [1]:
import torch
import torch.nn as nn

# Needed for parallel 
from collections import OrderedDict

# For training 
from network_architecture_v2 import MyBertForSequenceClassification

# For fine tuning
from datasets import load_dataset #, load_metric
from transformers import BertTokenizer
from transformers import Trainer, TrainingArguments
import numpy as np

In [12]:
# Load dataset
dataset = load_dataset('glue', 'sst2')

# I believe this is the tokenizer I used... 
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples['sentence'], padding="max_length", 
                     max_length=224, truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

# Load the parallel model

This involves a bit more code

In [13]:
checkpoint_0 = torch.load('bert-save-4/model_checkpoint_0_batch_idx=20000')
checkpoint_1 = torch.load('bert-save-4/model_checkpoint_1_batch_idx=20000')
checkpoint_2 = torch.load('bert-save-4/model_checkpoint_2_batch_idx=20000')
checkpoint_3 = torch.load('bert-save-4/model_checkpoint_3_batch_idx=20000')

In [14]:
keys_0 = checkpoint_0['model_state_dict'].keys()
keys_1 = checkpoint_1['model_state_dict'].keys()
keys_2 = checkpoint_2['model_state_dict'].keys()
keys_3 = checkpoint_3['model_state_dict'].keys()

In [15]:
# Ugh, this is so dumb
new_dict = OrderedDict()
keys_0 = checkpoint_0['model_state_dict'].keys()
counter = 0
for key in keys_0:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        if int(split[2]) > counter:
            counter = int(split[2])
            
        split.insert(3, 'layer')
        new_key = '.'.join(split[1:])
        new_dict[new_key] = checkpoint_0['model_state_dict'][key]
    else:
        new_key = key
        if 'close_nsp' in key:
            # print(key)
            split = key.split('.')
            split[0] = 'close_nn_nsp'
            new_key = '.'.join(split)
        if 'close_mlm' in key:
            # print(key)
            split = key.split('.')
            split[0] = 'close_nn_mlm'
            new_key = '.'.join(split)
        
        new_dict[new_key] = checkpoint_0['model_state_dict'][key]
        
print(counter)

# Now for the remaining parts? 
keys_1 = checkpoint_1['model_state_dict'].keys()
new_counter = 0
for key in keys_1:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        split[2] = str(int(split[2]) + counter + 1)
        split.insert(3, 'layer')

        if int(split[2]) > new_counter:
            new_counter = int(split[2])
        new_key = '.'.join(split[1:])
        # print(key, new_key)
        new_dict[new_key] = checkpoint_1['model_state_dict'][key]
    else:
        new_dict[key] = checkpoint_1['model_state_dict'][key]

print(new_counter)
counter = new_counter
new_counter = 0
keys_2 = checkpoint_2['model_state_dict'].keys()
for key in keys_2:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        split[2] = str(int(split[2]) + counter + 1)
        split.insert(3, 'layer')
        if int(split[2]) > new_counter:
            new_counter = int(split[2])
        new_key = '.'.join(split[1:])
        # print(key, new_key)
        new_dict[new_key] = checkpoint_2['model_state_dict'][key]
    else:
        new_dict[key] = checkpoint_2['model_state_dict'][key]

print(new_counter)
counter = new_counter
new_counter = 0
keys_3 = checkpoint_3['model_state_dict'].keys()
for key in keys_3:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        split[2] = str(int(split[2]) + counter + 1)
        split.insert(3, 'layer')
        if int(split[2]) > new_counter:
            new_counter = int(split[2])
        new_key = '.'.join(split[1:])
        # print(key, new_key)
        new_dict[new_key] = checkpoint_3['model_state_dict'][key]
    else:
        new_dict[key] = checkpoint_3['model_state_dict'][key]

32
64
96


In [16]:
model_parallel = torch.load('serialnet_bert_128', weights_only=False)
# model_parallel.load_state_dict(new_dict)

In [17]:
training_parallel = MyBertForSequenceClassification(model_parallel)

# With weights loaded, go ahead and train

In [18]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    adam_beta1=0.9,
    adam_beta2=0.988,
    adam_epsilon=1e-6,
    dataloader_drop_last=True,
    warmup_steps=100,
    weight_decay=1e-4,
    logging_dir='./logs',
    logging_steps=10,
    # evaluation_strategy="epoch",
)


In [19]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).astype(np.float32).mean().item()
    return {"accuracy": accuracy}

In [20]:
# Initialize the Trainer
trainer = Trainer(
    model=training_parallel,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics
)


In [21]:
trainer.train()

Step,Training Loss
10,0.735800
20,0.693300
30,0.707300
40,0.750000
50,0.714800
60,0.814900
70,0.742800
80,0.797900
90,0.746500
100,0.707100


KeyboardInterrupt: 

# Above was Sentiment analysis; move on to Cola



In [22]:
# Load dataset
dataset = load_dataset('glue', 'cola')

# I believe this is the tokenizer I used... 
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples['sentence'], padding="max_length", 
                     max_length=224, truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

In [23]:
training_parallel = MyBertForSequenceClassification(model_parallel)


In [27]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    adam_beta1=0.9,
    adam_beta2=0.988,
    adam_epsilon=1e-6,
    dataloader_drop_last=True,
    warmup_steps=100,
    weight_decay=1e-4,
    logging_dir='./logs',
    logging_steps=10,
    # evaluation_strategy="epoch",
)

# Initialize the Trainer
trainer = Trainer(
    model=training_parallel,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics
)



In [28]:
# For COLA
trainer.train()

Step,Training Loss
10,0.786900
20,0.612900
30,0.596200
40,0.679200
50,0.623800
60,0.734500
70,0.645400
80,0.686600
90,0.686500
100,0.731300


TrainOutput(global_step=3204, training_loss=0.6143522684493762, metrics={'train_runtime': 2871.5835, 'train_samples_per_second': 8.933, 'train_steps_per_second': 1.116, 'total_flos': 0.0, 'train_loss': 0.6143522684493762, 'epoch': 3.0})

In [31]:
trainer.evaluate()

{'eval_loss': 0.6233877539634705,
 'eval_accuracy': 0.6913461685180664,
 'eval_runtime': 37.3818,
 'eval_samples_per_second': 27.901,
 'eval_steps_per_second': 3.504,
 'epoch': 3.0}

## MRPC 

In [ ]:
training_parallel = MyBertForSequenceClassification(model_parallel)

In [ ]:
# Load dataset
dataset = load_dataset('glue', 'mrpc')

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["sentence1"], 
        examples["sentence2"], 
        padding="max_length", 
        truncation=True,
        max_length=224
    )
    
tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    adam_beta1=0.9,
    adam_beta2=0.988,
    adam_epsilon=1e-8,
    dataloader_drop_last=True,
    warmup_steps=5,
    weight_decay=1e-4,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
)

# Initialize the Trainer
trainer = Trainer(
    model=training_parallel,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics
)


In [ ]:
# For MRPC
trainer.train()